# SABR Replication Sanity Checks

This notebook complements the paper tables with a few model-level sanity checks that are not explicitly tabulated in the paper:

- `nu = 0` limit: the SABR simulator should collapse to the exact CEV model.
- `beta = 1, nu = 0` limit: the simulator should collapse to Black-Scholes / lognormal pricing.
- Martingale sanity check across maturities.
- `|rho| = 1` Islah edge case stability.
- Quick validation summary using the current experiment harness.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyfeng as pf

ROOT = Path.cwd()
if not (ROOT / 'sabr_replicate.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sabr_replicate import (
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    european_call_price,
    martingale_test,
    run_full_validation,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)


## 1. `nu = 0` should recover the exact CEV model

When `nu = 0`, the volatility is deterministic and SABR reduces to the CEV process. We compare Monte Carlo prices from our simulator with `pyfeng.Cev`'s exact pricing formula.

In [ ]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
cev_prices = pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'cev_price': cev_prices,
    'error': mc_prices - cev_prices,
})


## 2. `beta = 1, nu = 0` should recover Black-Scholes / lognormal pricing

In [ ]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
bsm_prices = pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'bsm_price': bsm_prices,
    'error': mc_prices - bsm_prices,
})


## 3. Martingale sanity check

The paper emphasizes martingale preservation. Here we run the same style of check over a range of maturities.

In [ ]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)


## 4. `|rho| = 1` Islah edge case stability

This is the edge case that previously lived only in a pytest. The notebook version makes it visible and easy to explain.

In [ ]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append({
        'beta': beta,
        'all_finite': bool(np.isfinite(terminal).all()),
        'all_nonnegative': bool((terminal >= 0.0).all()),
        'mean_terminal': float(np.mean(terminal)),
        'min_terminal': float(np.min(terminal)),
    })

pd.DataFrame(rows)


## 5. Quick validation summary

This reuses the repo's validation harness and prints the same statuses that the CLI would show.

In [ ]:
validation = run_full_validation(quick_mode=True)
summary = {
    'table1_status': validation['table1_status'],
    'table2_status': validation['table2_status'],
    'overall_conclusion': validation['overall_conclusion'],
    'replication_conclusion': validation['replication_conclusion'],
}
summary
